In [ ]:
import unicodedata as ud
import re
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

# --------- tiny text utils ----------
def normalize(s: str) -> str:
    if not isinstance(s, str): return ""
    s = ud.normalize("NFKC", s).lower()
    s = "".join(ch for ch in s if not ud.category(ch).startswith("P"))  # drop punctuation
    return re.sub(r"\s+", " ", s).strip()

def tokenize(s: str):
    return [t for t in normalize(s).split() if t]

def overlap_ratio(question: str, context: str) -> float:
    q = [t for t in tokenize(question) if len(t) >= 2]  # ignore 1-char tokens
    c = set(tokenize(context))
    if not q or not c:
        return 0.0
    qset = set(q)
    return len(qset & c) / len(qset)

# --------- heuristic classifier ----------
THRESHOLD = 0.25  # tune if you like; 0.15–0.25 are reasonable

def predict_answerable(question: str, context: str) -> int:
    return int(overlap_ratio(question, context) >= THRESHOLD)  # 1=answerable, 0=unanswerable

# --------- evaluation ----------
def eval_lang(rows):
    y_true, y_pred = [], []
    for r in rows:
        q, ctx = r.get("question"), r.get("context")
        if not q or not ctx:  # skip missing text
            continue
        y_true.append(1 if r.get("answerable") else 0)
        y_pred.append(predict_answerable(q, ctx))
    if not y_true:
        return None
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    return acc, f1, len(y_true)

def main():
    ds = load_dataset("coastalcph/tydi_xor_rc", split="validation")
    want = {"ar": "Arabic", "ko": "Korean", "te": "Telugu"}

    by_lang = {k: [] for k in want}
    for r in ds:
        lg = r.get("lang")
        if lg in by_lang:
            by_lang[lg].append(r)

    print(f"Word-overlap heuristic (threshold={THRESHOLD:.2f}) on TyDi XOR-RC validation")
    print(f"{'Language':10s}  {'Acc':>6s}  {'F1':>6s}  {'n':>6s}")
    for code, name in want.items():
        res = eval_lang(by_lang[code])
        if res is None:
            print(f"{name:10s}  (no rows)")
        else:
            acc, f1, n = res
            print(f"{name:10s}  {acc:6.3f}  {f1:6.3f}  {n:6d}")

if __name__ == "__main__":
    main()

Word-overlap heuristic (threshold=0.25) on TyDi XOR-RC validation
Language       Acc      F1       n
Arabic       0.130   0.011     415
Korean       0.056   0.006     356
Telugu       0.242   0.000     384
